# 🏷️ AutoTriage: NLP-Powered GitHub Issue Priority Detection

> **Author:** Jagdish Kollur  
> **Date:** March 2026  
> **Dataset:** sharjeelyunus/github-issues-dataset (114,073 GitHub issues)  
> **Model:** DistilBERT fine-tuned for priority classification  
> **Live Demo:** [Streamlit App](#) ← add link when deployed  
> **GitHub:** [https://github.com/jagdishkollur/autotriage-nlp](#) ← add wh 

---

## Table of Contents
1. [Business Understanding](#business-understanding)
2. [Problem Definition](#problem-definition)
3. [Data Loading & EDA](#eda)
4. [Preprocessing](#preprocessing)
5. [Baseline Model](#baseline)
6. [BERT Fine-Tuning](#bert)
7. [Threshold Optimization](#threshold)
8. [Explainability](#explainability)
9. [Conclusions](#conclusions)

In [1]:
# ── Environment & Library Versions ───────────────────────
# Pinning versions for reproducibility
import torch
import transformers
import datasets as hf_datasets
import numpy as np
import pandas as pd

print(f"PyTorch:        {torch.__version__}")
print(f"Transformers:   {transformers.__version__}")
print(f"HF Datasets:    {hf_datasets.__version__}")
print(f"NumPy:          {np.__version__}")
print(f"Pandas:         {pd.__version__}")
print(f"GPU Available:  {torch.cuda.is_available()}")
print(f"GPU Name:       {torch.cuda.get_device_name(0)}")

PyTorch:        2.10.0+cu128
Transformers:   5.0.0
HF Datasets:    4.8.3
NumPy:          2.0.2
Pandas:         2.3.3
GPU Available:  True
GPU Name:       Tesla T4


---
## 1. Business Understanding
*Goal: Define who has this problem and what success looks like — before touching any data.*



## Problem Statement
Open source maintainers, engineering leads, and project managers spend 2-3 hours daily manually reading 
and sorting GitHub issues — causing critical production bugs to sit unattended for days while buried 
under feature requests and questions. This leads to system downtime, damaged project reputation,
and maintainer burnout. Without automation, there is no scalable way to surface what needs immediate 
attention. AutoTriage solves this by automatically predicting the priority of GitHub issues — high, medium, or low — using only the issue title and description text  — reducing manual triage time from 2-3 hours to 15-20 minutes and ensuring critical bugs are never buried.

## Who Has This Problem
- Open source maintainers managing 50-100 new issues daily
- Engineering team leads who need to surface critical bugs immediately
- Project managers tracking sprint priorities across large repos

## Cost of Getting It Wrong
- False Negative (missing a critical bug): Production downtime, 
  user impact, reputation damage — most costly error
- False Positive (over-flagging low priority): 30 min wasted 
  investigation — acceptable tradeoff

## Success Metrics (defined before training)
- Macro F1 > 0.75 (penalizes ignoring minority classes)
- High priority class Recall > 0.85 (missing urgent issues is costly)
- Inference time < 2 seconds per issue
- Baseline comparison: TF-IDF + LogReg before BERT

---
## 2. Problem Definition
*Goal: Define inputs, outputs, and problem type with mathematical precision.*


## Problem Definition — Final (Updated After Data Investigation)

**Input:**
Issue Title + Issue Body combined as single text
Format: "[title] [SEP] [body]"

**Output — Priority:**
high / medium / low

**Problem Type:**
Single-output multi-class text classification

**Why only Priority?**
Original scope included Issue Type and Severity.
Data investigation revealed:
- labels column: 40,438 unique combinations — unusable
- severity column: 58% labeled Critical — ML-generated noise
Priority was the only column with clean, reliable labels.

---
## 3. Data Investigation
*Goal: Understand the data quality and confirm our label assumptions.*

In [2]:
# ── Load Dataset from HuggingFace ────────────────────────
# sharjeelyunus/github-issues-dataset — 114,073 GitHub issues
# from top 100 repositories
from datasets import load_dataset
ds = load_dataset("sharjeelyunus/github-issues-dataset")
df = ds['train'].to_pandas()
print(df['priority'].value_counts())
print(df['severity'].value_counts())

priority
low       101937
medium     10132
high        2004
Name: count, dtype: int64
severity
Critical    66491
Minor       29769
Major       17813
Name: count, dtype: int64


In [3]:
# ── Qualitative Label Inspection ─────────────────────────
# Sample 20 "Critical" severity issues
# Goal: verify if Critical label reflects actual critical content
critical_issues = df[df['severity'] == 'Critical'][['title','body','severity']].sample(20)
print(critical_issues['title'].tolist())

['Disable Python mode (torch dispatch mode) inside of mode-induced __torch_dispatch__call', ' "HTTP Error 401: Unauthorized" if passing in creds or useragent\\cookies', 'Typescript 5 decorators do not build', 'provider/node.vim hang on exit', "encoding/asn1: marshaling GeneralizedTime and UTCTime don't follow DER restriction", 'Option button giving false item count C#', 'Encoding empty string to base64 reports an error `Condition "ret.is_empty()" is true. Returning: ret`, but should not', 'Product Icon Theme crash on Linux', 'Unknown builtin op: torchvision::nms in LibTorch', '[NEXT-780] The same library is imported as both ESM and CommonJS module', 'Create a theme flag that turns on automated platform adaptations for all applicable Material widgets', 'Preventing overlay of sheet component', 'PyTorch gets stuck when using an NVLink/A6000 and more than two GPUs', 'Use of deprecated webpack DevServer onBeforeSetupMiddleware and onAfterSetupMiddleware options', 'Add a flag which prioritie

In [4]:
# ── Severity vs Priority Cross-Tabulation ─────────────────
# Key question: are severity and priority correlated?
# If not — one or both columns are unreliable labels
print(df['priority'].unique())
print(pd.crosstab(df['severity'], df['priority']))

['medium' 'low' 'high']
priority  high    low  medium
severity                     
Critical  1893  57525    7073
Major       98  14967    2748
Minor       13  29445     311


**Observation 1 — Critical + Low = 57,525 issues (50% of dataset)**
In real-world engineering, a Critical severity issue should 
never be Low priority. This combination appearing 57,525 times 
indicates the labels are unreliable.

**Observation 2 — Severity and Priority were labeled independently**
A reliable dataset would show correlation between these two columns
— critical issues should skew toward high priority. The broken 
correlation confirms these labels were generated by two separate 
ML models with no awareness of each other.


### 💡 Decision — Drop Severity Column
After investigating the data, 58% of issues were labeled 
Critical regardless of actual content — indicating the 
severity labels were unreliable ML-generated noise.

**Decision:** Drop severity as a target output.
Focus solely on predicting priority.
This is a data quality decision, not a modeling shortcut.

In [5]:
# ── Issue Type Label Analysis ─────────────────────────────
# Checking if labels column can be used as issue type target
print(df['labels'].value_counts().head(20))
print(f"\nTotal unique label values: {df['labels'].nunique()}")
print(f"Total rows: {len(df)}")
combo = df['labels'].str.contains(',', na=False).sum()
print(f"Rows with combined labels: {combo}")

labels
bug                                    2380
NeedsInvestigation                     1869
Suggestion,Awaiting More Feedback      1195
bug-report                             1137
enhancement                            1072
Needs-Triage                            940
site-support-request                    691
Issue-Bug,Needs-Triage                  636
Bug                                     613
oncall: jit                             607
needs triage,issue: bug report          557
A-diagnostics,T-compiler                543
Proposal                                518
feature request                         511
NeedsInvestigation,compiler/runtime     478
bug,topic:editor                        455
request                                 420
Suggestion,In Discussion                406
bug,needs triage                        398
Needs Investigation                     371
Name: count, dtype: int64

Total unique label values: 40438
Total rows: 114073
Rows with combined labels: 92060


In [6]:
# ── Final Target Confirmation ─────────────────────────────
# After dropping severity and labels — priority is our sole target
# Documenting final class distribution and imbalance ratio
print("FINAL TARGET DISTRIBUTION:")
print(df['priority'].value_counts())
low = df['priority'].value_counts()['low']
med = df['priority'].value_counts()['medium']
high = df['priority'].value_counts()['high']
print(f"\nClass ratio (low:medium:high): {low//high}:{med//high}:1")





FINAL TARGET DISTRIBUTION:
priority
low       101937
medium     10132
high        2004
Name: count, dtype: int64

Class ratio (low:medium:high): 50:5:1


## Phase 2 Complete — Key Decisions Made

| Decision | Reason |
|----------|--------|
| Dropped severity | 58% labeled Critical — unreliable ML-generated labels |
| Dropped issue type | 40,438 unique label combinations — unusable |
| Kept priority only | 3 clean classes with real business value |
| Metric: Macro F1 | Accuracy misleading with 50:5:1 imbalance |
| Focus: high recall | Missing high priority issue is most costly error |

### Final Target
priority → high (1.7%) / medium (8.9%) / low (89.4%)
Class imbalance ratio: 50:5:1

---
## 4. Exploratory Data Analysis (EDA)
*Goal: Understand the text characteristics of each priority 
class, identify data quality issues, and make all preprocessing 
decisions based on evidence — not assumptions.*

In [7]:
# ── Null & Empty Value Check ──────────────────────────────
# Goal: understand how much data is missing in our input columns
# Missing body text is common — we need a strategy to handle it

print("=" * 45)
print("NULL VALUES")
print("=" * 45)
print(df[['title', 'body', 'priority']].isnull().sum())

print("\n" + "=" * 45)
print("EMPTY STRINGS")
print("=" * 45)
print(f"Empty title:  {(df['title'] == '').sum()}")
print(f"Empty body:   {(df['body'] == '').sum()}")

print("\n" + "=" * 45)
print("PERCENTAGE MISSING BODY")
print("=" * 45)
missing_body = df['body'].isnull().sum() + (df['body'] == '').sum()
print(f"{missing_body / len(df) * 100:.2f}% of issues have no body text")

NULL VALUES
title         0
body        133
priority      0
dtype: int64

EMPTY STRINGS
Empty title:  0
Empty body:   85

PERCENTAGE MISSING BODY
0.19% of issues have no body text


### 💡 Finding — Missing Values
- 133 null body values + 85 empty strings = 218 total (0.19%)
- Title has zero missing values — perfectly clean
- Priority has zero missing values — target is intact

**Decision:** Drop all 218 rows with missing/empty body text.
Justification: 0.19% data loss is negligible. Imputation 
on text data is unreliable — a fake body adds noise, not signal.

In [8]:
# ── Text Length Analysis ──────────────────────────────────
# Goal: understand how long titles and bodies are
# This directly determines max_length for our BERT tokenizer
# Setting max_length too low = truncating important information
# Setting max_length too high = wasting GPU memory and time

df['title_len'] = df['title'].str.len()
df['body_len'] = df['body'].fillna('').str.len()

print("=" * 45)
print("TITLE LENGTH (characters)")
print("=" * 45)
print(df['title_len'].describe().round(1))

print("\n" + "=" * 45)
print("BODY LENGTH (characters)")
print("=" * 45)
print(df['body_len'].describe().round(1))

TITLE LENGTH (characters)
count    114073.0
mean         62.4
std          26.5
min           1.0
25%          45.0
50%          59.0
75%          76.0
max         936.0
Name: title_len, dtype: float64

BODY LENGTH (characters)
count    114073.0
mean       2532.3
std        6065.5
min           0.0
25%         675.0
50%        1310.0
75%        2479.0
max      255646.0
Name: body_len, dtype: float64


### 💡 Finding — Text Length Distribution

**Title:** Clean and consistent
- Average: 62 characters (~10 words)
- Max: 936 characters — a few outliers exist
- 75% of titles under 76 characters

**Body:** Heavily skewed with extreme outliers
- Average: 2,532 characters
- Median: 1,310 characters  
- Max: 255,646 characters — someone pasted an entire log file
- Std dev (6,065) > Mean (2,532) — confirms heavy-tailed distribution

**Key insight:** Cannot use mean for max_length decision.
Must use percentile-based approach to avoid wasting GPU memory
on extreme outliers while preserving most information.

In [9]:
# ── Text Length by Priority Class ────────────────────────
# Goal: do high priority issues use more text than low priority?
# If yes — text length itself becomes a signal for the model
# If no — length is not a useful feature

print("=" * 45)
print("AVERAGE TITLE LENGTH BY PRIORITY")
print("=" * 45)
print(df.groupby('priority')['title_len'].mean().round(1))

print("\n" + "=" * 45)
print("AVERAGE BODY LENGTH BY PRIORITY")
print("=" * 45)
print(df.groupby('priority')['body_len'].mean().round(1))

print("\n" + "=" * 45)
print("MEDIAN BODY LENGTH BY PRIORITY")
print("=" * 45)
print(df.groupby('priority')['body_len'].median().round(1))

AVERAGE TITLE LENGTH BY PRIORITY
priority
high      53.3
low       62.8
medium    60.2
Name: title_len, dtype: float64

AVERAGE BODY LENGTH BY PRIORITY
priority
high      2326.9
low       2537.5
medium    2520.5
Name: body_len, dtype: float64

MEDIAN BODY LENGTH BY PRIORITY
priority
high      1317.5
low       1295.0
medium    1445.0
Name: body_len, dtype: float64


### 💡 Finding — Text Length vs Priority

| Priority | Avg Title | Avg Body | Median Body |
|----------|-----------|----------|-------------|
| high     | 53 chars  | 2,327    | 1,318       |
| medium   | 60 chars  | 2,521    | 1,445       |
| low      | 63 chars  | 2,538    | 1,295       |

**Key insight:** Text length has no meaningful correlation 
with priority. The model cannot rely on length as a shortcut.
It must understand the actual content — justifying the use 
of a semantic model like DistilBERT over simpler approaches.

In [10]:
# ── 95th Percentile — BERT max_length Decision ───────────
# Goal: find the right max_length for our tokenizer
# BERT tokenizes into subword tokens — roughly 1 token per 4 chars
# We use 95th percentile to cover most issues without
# wasting GPU memory on extreme outliers like 255k char issues

import numpy as np

df['combined_len'] = df['title_len'] + df['body_len']

print("=" * 45)
print("COMBINED TEXT LENGTH (title + body)")
print("=" * 45)
print(df['combined_len'].describe().round(1))

print("\n" + "=" * 45)
print("PERCENTILE ANALYSIS")
print("=" * 45)
for p in [50, 75, 90, 95, 99]:
    chars = np.percentile(df['combined_len'], p)
    tokens = int(chars / 4)
    print(f"  {p}th percentile: {chars:.0f} chars → ~{tokens} tokens")

print("\n" + "=" * 45)
print("MAX_LENGTH RECOMMENDATION")
print("=" * 45)
p95_tokens = int(np.percentile(df['combined_len'], 95) / 4)
recommended = min(p95_tokens, 512)
print(f"95th percentile tokens: {p95_tokens}")
print(f"BERT hard limit:        512")
print(f"Recommended max_length: {recommended}")
print(f"\nThis covers 95% of issues with no truncation")
print(f"The remaining 5% will be truncated at {recommended} tokens")

COMBINED TEXT LENGTH (title + body)
count    114073.0
mean       2594.6
std        6067.5
min           9.0
25%         734.0
50%        1375.0
75%        2546.0
max      255723.0
Name: combined_len, dtype: float64

PERCENTILE ANALYSIS
  50th percentile: 1375 chars → ~343 tokens
  75th percentile: 2546 chars → ~636 tokens
  90th percentile: 4872 chars → ~1218 tokens
  95th percentile: 7522 chars → ~1880 tokens
  99th percentile: 23208 chars → ~5802 tokens

MAX_LENGTH RECOMMENDATION
95th percentile tokens: 1880
BERT hard limit:        512
Recommended max_length: 512

This covers 95% of issues with no truncation
The remaining 5% will be truncated at 512 tokens


In [11]:
# ── Qualitative Analysis — Read Real Examples ────────────
# Goal: read actual high priority issues with human eyes
# No algorithm can replace this — you must understand
# what makes a high priority issue LOOK different in text

import random
random.seed(42)

for priority in ['high', 'medium', 'low']:
    print("=" * 55)
    print(f"PRIORITY: {priority.upper()} — 3 RANDOM EXAMPLES")
    print("=" * 55)
    
    samples = df[df['priority'] == priority][['title', 'body'] ].dropna().sample(3, random_state=42)
    
    for i, (_, row) in enumerate(samples.iterrows(), 1):
        print(f"\n--- Example {i} ---")
        print(f"TITLE: {row['title']}")
        print(f"BODY:  {str(row['body'])[:200]}...")
        print()

PRIORITY: HIGH — 3 RANDOM EXAMPLES

--- Example 1 ---
TITLE: proposal: spec: generic parameterization of array sizes
BODY:  See detailed design [here][detailed proposal]. The recent acceptance of #43651, we can now begin discussing filling in the gaps and omissions of that proposal. This issue provides a proposal to fulfil...


--- Example 2 ---
TITLE: Keyboard mappings with `setxkbmap` on Linux not working
BODY:  - VSCode Version: 1.11.0
- OS Version: Ubuntu 16.04

I have my Caps Lock key bound to Escape using `setxkbmap` on Linux. This worked fine till 1.10, but broke with 1.11. At first I thought it was a...


--- Example 3 ---
TITLE: NodeMaterial: Support MaterialX Standard Nodes
BODY:  I'd like to suggest that we work toward supporting equivalents of the MaterialX Standard Nodes within NodeMaterial. That is not necessarily to say that everything needs to be named the same, but ideal...

PRIORITY: MEDIUM — 3 RANDOM EXAMPLES

--- Example 1 ---
TITLE: DashCase and CamelCase intrinsic

### 💡 Finding — Qualitative Text Analysis

After manually reading 9 examples across all priority classes:

**Patterns observed:**

High priority language tends to include:
- Regression signals: "worked in X, broke in Y"
- Failure language: "not working", "broken", "crash"
- Specific version references indicating a breaking change

Low priority language tends to include:
- Proposal language: "I'd like to suggest", "it would be great"
- Feature requests: "please add", "consider implementing"
- Enhancement framing: "would be useful if"

**Key finding:** Labels are noisy but not random.
Some mislabelings exist (feature proposals labeled high,
real bugs labeled low) but enough signal exists for
a BERT model to learn meaningful patterns at scale.

**Implication:** Model performance ceiling may be limited
by label noise. Honest evaluation is essential.
Macro F1 > 0.75 remains our target — adjusted expectations
if label noise prevents this.

In [12]:
# ── Word Frequency by Priority Class ─────────────────────
# Goal: find words that appear disproportionately in 
# high priority vs low priority issues
# This confirms there IS learnable signal in the text

from collections import Counter
import re

def get_top_words(df, priority, n=15):
    # Get all text for this priority class
    texts = df[df['priority'] == priority]['title'].fillna('')
    
    # Combine all titles into one string
    all_text = ' '.join(texts).lower()
    
    # Remove punctuation and split into words
    words = re.findall(r'\b[a-z]{4,}\b', all_text)
    
    # Remove common stop words
    stopwords = {'this', 'that', 'with', 'from', 'have',
                 'when', 'will', 'does', 'should', 'using',
                 'after', 'been', 'some', 'into', 'would',
                 'there', 'their', 'about', 'which', 'issue',
                 'error', 'support', 'feature', 'request'}
    
    words = [w for w in words if w not in stopwords]
    return Counter(words).most_common(n)

print("=" * 45)
print("TOP WORDS IN HIGH PRIORITY TITLES")
print("=" * 45)
for word, count in get_top_words(df, 'high'):
    print(f"  {word:<20} {count}")

print("\n" + "=" * 45)
print("TOP WORDS IN LOW PRIORITY TITLES")
print("=" * 45)
for word, count in get_top_words(df, 'low'):
    print(f"  {word:<20} {count}")

TOP WORDS IN HIGH PRIORITY TITLES
  proposal             122
  tracking             117
  allow                108
  type                 94
  spec                 60
  types                48
  runtime              47
  file                 45
  code                 45
  build                45
  windows              43
  files                43
  react                41
  json                 32
  option               31

TOP WORDS IN LOW PRIORITY TITLES
  type                 3931
  torch                3129
  doesn                2951
  file                 2739
  work                 2447
  build                2415
  windows              2207
  editor               2150
  code                 1976
  function             1970
  fails                1921
  test                 1884
  cannot               1866
  flutter              1757
  text                 1731


In [13]:
# ── Verify the Inversion — Keyword Check ─────────────────
# Count how many high priority issues contain proposal/tracking
# vs how many contain failure language

high_df = df[df['priority'] == 'high']
low_df = df[df['priority'] == 'low']

proposal_words = ['proposal', 'tracking', 'spec', 'allow']
failure_words = ['crash', 'broken', 'fails', 'cannot', 
                 'regression', 'not working']

print("=" * 50)
print("HIGH PRIORITY — keyword presence")
print("=" * 50)
for word in proposal_words + failure_words:
    count = high_df['title'].str.lower().str.contains(
        word, na=False).sum()
    pct = count / len(high_df) * 100
    print(f"  '{word}': {count} issues ({pct:.1f}%)")

print("\n" + "=" * 50)
print("LOW PRIORITY — keyword presence")
print("=" * 50)
for word in proposal_words + failure_words:
    count = low_df['title'].str.lower().str.contains(
        word, na=False).sum()
    pct = count / len(low_df) * 100
    print(f"  '{word}': {count} issues ({pct:.1f}%)")

HIGH PRIORITY — keyword presence
  'proposal': 122 issues (6.1%)
  'tracking': 117 issues (5.8%)
  'spec': 98 issues (4.9%)
  'allow': 116 issues (5.8%)
  'crash': 17 issues (0.8%)
  'broken': 7 issues (0.3%)
  'fails': 13 issues (0.6%)
  'cannot': 30 issues (1.5%)
  'regression': 2 issues (0.1%)
  'not working': 14 issues (0.7%)

LOW PRIORITY — keyword presence
  'proposal': 1105 issues (1.1%)
  'tracking': 741 issues (0.7%)
  'spec': 2265 issues (2.2%)
  'allow': 1980 issues (1.9%)
  'crash': 1849 issues (1.8%)
  'broken': 741 issues (0.7%)
  'fails': 1918 issues (1.9%)
  'cannot': 1847 issues (1.8%)
  'regression': 390 issues (0.4%)
  'not working': 1102 issues (1.1%)


### 💡 Finding — Label Inversion Confirmed

Keyword analysis confirms systematic label noise:

| Keyword | High Priority | Low Priority |
|---------|--------------|--------------|
| 'proposal' | 6.1% | 1.1% |
| 'tracking' | 5.8% | 0.7% |
| 'crash' | 0.8% | 1.8% |
| 'fails' | 0.6% | 1.9% |
| 'regression' | 0.1% | 0.4% |

**Root cause hypothesis:**
Certain repositories (language spec repos like Rust, 
TypeScript, Go) labeled roadmap/tracking issues as 
high priority — inflating high priority with 
non-urgent planning issues.

**Decision: Reframe the project**
Instead of predicting noisy ML-generated priority labels,
build a rule-assisted classifier that uses both text 
signals AND the existing priority labels as weak supervision.
The model learns real urgency patterns from text — not 
the inverted noise in the labels.

In [14]:
# ── Check Which Repos Dominate High Priority ──────────────
# Hypothesis: spec/language repos are inflating high priority
# with planning issues

print("=" * 50)
print("TOP REPOS IN HIGH PRIORITY ISSUES")
print("=" * 50)
print(df[df['priority']=='high']['repo'].value_counts().head(15))

print("\n" + "=" * 50)
print("TOP REPOS IN LOW PRIORITY ISSUES")  
print("=" * 50)
print(df[df['priority']=='low']['repo'].value_counts().head(15))

TOP REPOS IN HIGH PRIORITY ISSUES
repo
vscode              305
go                  298
TypeScript          216
rust                186
flutter             132
kubernetes          114
angular              94
pytorch              65
create-react-app     58
PowerToys            58
react                42
react-native         40
godot                36
terminal             34
node                 30
Name: count, dtype: int64

TOP REPOS IN LOW PRIORITY ISSUES
repo
pytorch                   13722
flutter                   12185
godot                     10608
rust                       8923
go                         7654
vscode                     6429
PowerToys                  5365
TypeScript                 4722
opencv                     2439
next.js                    2319
stable-diffusion-webui     2128
youtube-dl                 1846
storybook                  1748
deno                       1622
material-ui                1508
Name: count, dtype: int64


In [15]:
# ── Filter to Application Repos ───────────────────────────
# Remove language/spec repos where high priority = roadmap
# Keep application repos where high priority = urgent bug

app_repos = ['pytorch', 'flutter', 'godot', 'react-native',
             'react', 'angular', 'next.js', 'material-ui',
             'stable-diffusion-webui', 'opencv', 'storybook',
             'deno', 'youtube-dl', 'kubernetes', 'node',
             'create-react-app']

df_clean = df[df['repo'].isin(app_repos)].copy()

print("=" * 45)
print("FILTERED DATASET SIZE")
print("=" * 45)
print(f"Original:  {len(df):,} rows")
print(f"Filtered:  {len(df_clean):,} rows")
print(f"Removed:   {len(df) - len(df_clean):,} rows")

print("\n" + "=" * 45)
print("NEW PRIORITY DISTRIBUTION")
print("=" * 45)
print(df_clean['priority'].value_counts())

print("\n" + "=" * 45)
print("NEW CLASS RATIO")
print("=" * 45)
low = df_clean['priority'].value_counts()['low']
high = df_clean['priority'].value_counts()['high']
med = df_clean['priority'].value_counts().get('medium', 0)
print(f"low:medium:high = {low//high}:{med//high}:1")

FILTERED DATASET SIZE
Original:  114,073 rows
Filtered:  61,202 rows
Removed:   52,871 rows

NEW PRIORITY DISTRIBUTION
priority
low       55739
medium     4743
high        720
Name: count, dtype: int64

NEW CLASS RATIO
low:medium:high = 77:6:1


### 💡 Finding — Filtering Reduced High Priority Too Much

Filtering to application repos only:
- Reduced dataset from 114,073 → 61,202 rows
- Reduced high priority from 2,004 → 720 examples
- Worsened imbalance from 50:1 → 77:1

720 high priority examples is below the minimum threshold
for reliable BERT fine-tuning (~1,000 per class needed).

**Decision: Revert to full dataset.**
Label noise is documented and accepted as a real-world
constraint. Model learns generalised priority signal
across both repo type definitions.

In [16]:
# ── Revert to Full Dataset ────────────────────────────────
# Filtering reduced high priority from 2,004 to 720 rows
# 720 examples is insufficient for reliable BERT fine-tuning
# Decision: use full dataset with documented label limitations

df_final = df.copy()

# Drop 218 rows with missing or empty body text
df_final = df_final[
    df_final['body'].notna() & 
    (df_final['body'] != '')
].copy()

# Reset index after dropping rows
df_final = df_final.reset_index(drop=True)

print(f"Original dataset:  {len(df):,} rows")
print(f"Rows dropped:      {len(df) - len(df_final):,} (missing body)")
print(f"Final dataset:     {len(df_final):,} rows")
print()
print("FINAL TARGET DISTRIBUTION:")
print(df_final['priority'].value_counts())
print()
print(f"Class ratio: {df_final['priority'].value_counts()['low'] // df_final['priority'].value_counts()['high']}:1")



 



Original dataset:  114,073 rows
Rows dropped:      218 (missing body)
Final dataset:     113,855 rows

FINAL TARGET DISTRIBUTION:
priority
low       101735
medium     10120
high        2000
Name: count, dtype: int64

Class ratio: 50:1


---
## EDA Complete — All Findings Summary

| Finding | Impact | Decision |
|---------|--------|----------|
| 0.19% missing body | Negligible data loss | Drop 218 rows |
| Text length heavily skewed | Max 255k chars exists | Use title + first 200 chars of body |
| max_length for tokenizer | 95th percentile = 1,880 tokens | Set max_length=256 (title-focused) |
| Text length ≠ priority signal | Model cannot use length as shortcut | BERT semantic understanding needed |
| Labels systematically noisy | Proposals labeled high priority | Document honestly, use full data |
| Root cause: repo type mismatch | Language vs app repos differ | Transparent limitation in README |
| Filtering not viable | 720 high priority too few | Revert to full 114,073 rows |

### Final Setup for Phase 4
- **Input:** title + first 200 chars of body
- **Target:** priority (high / medium / low)
- **Dataset:** 113,855 rows (after dropping 218 missing body)
- **max_length:** 256 tokens
- **Expected Macro F1 ceiling:** ~0.65 due to label noise
- **Primary metric:** Macro F1
- **Secondary metric:** High priority Recall > 0.85
```



